<div style="padding: 20px; background: linear-gradient(90deg, #FDC830 0%, #F37335 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🤖 Module 6.4: Self-Querying Retriever</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Translating Natural Language into Metadata Database Filters.</p>
</div>

---

## 1. The Metadata Filtering Problem
In Module 4, we learned that we can filter Vector Databases using Metadata (e.g., `filter={"year": 2005}`). 
But users don't type JSON dictionaries into chat boxes. They type natural language: 
> *"Find me some sci-fi movies directed by Nolan after 2010."*

## 2. Self-Query Retriever
Self-Querying solves this. When the user types a prompt, an LLM intercepts it and splits it into two parts:
1. **The Semantic Query**: `"sci-fi movies"`
2. **The Metadata Filter**: `{"director": "Christopher Nolan", "year": 2010}`

We will build this flow natively using JSON parsing to automatically filter Chroma!

### Course alignment and free-first stack

- Covers: Self-query retrieval that converts natural language into metadata filters.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os
import json
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

docs = [
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose", metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"}),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream", metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2}),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams", metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6}),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them", metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3}),
    Document(page_content="Toys come alive and have an existential crisis", metadata={"year": 1995, "genre": "animated"})
]

embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="self_query_demo2")

## 3. Defining the Schema & Logic
We tell the LLM exactly what metadata fields exist, and force it to output JSON.

In [ ]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    prompt = PromptTemplate.from_template(
        """You are an expert database query constructor.
        Given a user's question, extract the semantic search query and any exact metadata filters.
        Available metadata fields:
        - year (integer): The release year
        - director (string): The movie director
        
        Output ONLY a valid JSON object with 'query' (string) and 'filter' (dictionary).
        If there are no filters, make 'filter' an empty dictionary.
        Do not output any markdown formatting or backticks, just the raw JSON.
        
        User Question: {question}
        JSON Output:"""
    )
    
    chain = prompt | llm
    
    def self_query(user_input):
        # 1. Ask LLM to build the query JSON
        res = chain.invoke({"question": user_input})
        
        try:
            parsed = json.loads(res.content)
            search_query = parsed.get("query", user_input)
            db_filter = parsed.get("filter", {})
        except:
            search_query = user_input
            db_filter = {}
            
        print(f"\nUser: {user_input}")
        print(f"[LLM Parsed] Query: '{search_query}', Filter: {db_filter}")
        
        # 2. Execute against Chroma using the extracted filter
        results = vectorstore.similarity_search(search_query, filter=db_filter if db_filter else None)
        for d in results:
            print(f" → {d.page_content} (Metadata: {d.metadata})")
            
    self_query("I want to watch a movie directed by Christopher Nolan")
    self_query("Are there any movies from the year 1995?")
else:
    print("Please provide a GROQ_API_KEY in your .env to see this run.")